In [ ]:
import pandas as pd

In [ ]:
# Função para filtrar a tabela segundo alguns parametros
def filtrar_tabela(d, campo, *args):
    filtro = [*args]

    return d[d[campo].isin(filtro)]

# Modificando o __repr__ dos dataframes para ter uma exibição mais organizada com título da tabela
class PersonalizarDF(pd.DataFrame):
    def _repr_html_(self):
        html = self.rename_axis(columns=None).to_html(
            index=False,
            notebook=True,
        )
        return f"<h3> {self.columns.name}</h3>{html}"

    # Com o constructor, todos os dataframes que forem criados a partir deste serão desta classe
    @property
    def _constructor(self):
        return PersonalizarDF



In [ ]:
# Limpeza dos dados

arq = "agrobd.csv"

# Colocando os anos e os produtos em listas para facilitar o uso depois
anos = list(range(2021, 2026))
produtos = ["Algodão herbáceo (em caroço)", "Café (em grão) Total",
            "Cana-de-açúcar", "Milho (em grão)", "Soja (em grão)"]

# Linha (a partir de 0) onde começam os dados de cada bloco no dataset
blocos = {
    "Área plantada ou destinada à colheita (ha)": 4,
    "Área colhida (ha)": 37,
    "Quantidade produzida (t)": 70,
    "Rendimento médio (kg/ha)": 103,
    "Valor da produção (mil R$)": 137,
}

# Organizando os anos e os produtos em colunas da tabela
cols = ["Local"] + [(a, p) for a in anos for p in produtos]

# Montando o dataframe por categoria
df = []
for var, inicio in blocos.items():
    df_temp = pd.read_csv(
        arq, sep=",", encoding="utf-8-sig",
        skiprows=inicio, nrows=28, header=None, names=cols,
        na_values=["-", "..", "...", "X"],
    )
    # Colocando o ano e os produtos como valores de variáveis
    df_temp = df_temp.melt(id_vars="Local", var_name="ano_produto", value_name="valor")

    # Separando ano e produto em colunas diferentes
    df_temp[["Ano", "produto"]] = pd.DataFrame(df_temp["ano_produto"].tolist(), index=df_temp.index)

    # Colocando os produtos como colunas
    df_temp = df_temp.pivot(index=["Local", "Ano"], columns="produto", values="valor").reset_index()
    df_temp.columns.name = None
    df_temp[produtos] = df_temp[produtos].fillna(0).astype("Int32")

    df_temp = df_temp.rename_axis(index=None, columns=var)
    df.append(PersonalizarDF(df_temp))

# Separando os dataframes por categoria
df_plantada, df_colhida, df_produzida, df_rendimento, df_valor = df

# Testando
display(df_plantada)

Local,Ano,Algodão herbáceo (em caroço),Café (em grão) Total,Cana-de-açúcar,Milho (em grão),Soja (em grão)
Acre,2021,0,1055,503,34105,6185
Acre,2022,0,1062,480,40460,6570
Acre,2023,0,993,465,40831,12010
Acre,2024,0,1115,418,37525,17578
Acre,2025,0,1931,419,39188,17478
Alagoas,2021,960,0,291462,55778,3312
Alagoas,2022,630,0,279705,54019,3686
Alagoas,2023,32,0,290003,54358,5796
Alagoas,2024,33,2,274550,52459,5054
Alagoas,2025,34,2,265573,55518,2260


In [ ]:
# Exportando base de dados tratada
df_plantada.to_csv("df_plantada.csv", index=False)
df_colhida.to_csv("df_colhida.csv", index=False)
df_produzida.to_csv("df_produzida.csv", index=False)
df_rendimento.to_csv("df_rendimento.csv", index=False)
df_valor.to_csv("df_valor.csv", index=False)

with pd.ExcelWriter("agrobd_tratada.xlsx") as writer:
    df_plantada.to_excel(writer, sheet_name="Área plantada")
    df_colhida.to_excel(writer, sheet_name="Área colhida")
    df_produzida.to_excel(writer, sheet_name="Quantidade produzida")
    df_rendimento.to_excel(writer, sheet_name="Rendimento médio")
    df_valor.to_excel(writer, sheet_name="Valor da produção")